# 02 검색 근거와 출처가 있는 답변 — 검색한 원문이 어떻게 답변과 출처가 되는가

완성된 코드를 함께 실행합니다(45분). 같은 질문에 대해 다음 네 단계를 차례로 출력합니다.

`① 검색된 원문 → ② LLM에 전달되는 질문과 근거 → ③ 답변 → ④ 인용문과 실제 출처`

- 문서: 삼일PwC 『반도체 산업 Business Guidebook』(2025.12) **책자 6쪽 = PDF 8번째 페이지** ‘2. 반도체 Value Chain Overview (8대 공정)’ · [발행처 원문(8번째 페이지)](https://www.pwc.com/kr/ko/insights/industry-focus/samilpwc_semicon-guidebook.pdf#page=8)
- 원본 PDF는 함께 배포하지 않습니다. 노트북 옆 `data/samilpwc_semicon-guidebook.pdf`가 없으면 발행처 주소에서 내려받습니다. 내려받을 수 없는 환경이면 위 링크에서 직접 받아 그 위치에 두세요.
- 앞 챕터를 실행하지 않아도 됩니다. 새 커널에서 첫 셀부터 차례로 실행합니다.
- 채팅 모델 gpt-6-luna와 임베딩 모델 text-embedding-3-small을 씁니다(키 하나, 주소 두 개).

In [ ]:
# 현재 Python 3.12 커널에 실습 패키지를 설치합니다(처음 한 번).
# 설치 후 커널 재시작 안내가 나오면 재시작하고 다음 셀부터 실행하세요.
%pip install langchain-core==1.8.4 langchain-openai==1.8.5 langchain-chroma==1.2.0 chromadb==1.5.9 openai==3.59.0 pypdf==7.1.1

In [ ]:
# ▶ 설정 셀 — 따옴표 안의 값만 바꾸세요. 모델명은 확정값이라 바꾸지 않습니다.
# 키가 든 이 셀은 화면 공유·캡처하거나 AI 도우미에 붙여넣지 않습니다. 파일을 공유할 때는 자리표시 문자열로 되돌립니다.
API_KEY = "여기에_수업용_API_키_입력"
BASE_URL = "여기에_채팅용_BASE_URL_입력"                # 안내받은 채팅 모델 주소(https://...)
MODEL_NAME = "gpt-6-luna"                              # 채팅·도구 호출·에이전트 모델(확정값)
EMBEDDING_BASE_URL = "여기에_임베딩용_BASE_URL_입력"    # 안내받은 임베딩 모델 주소(https://..., 채팅 주소와 다름)
EMBEDDING_MODEL_NAME = "text-embedding-3-small"        # 문서·질문 임베딩 모델(확정값)

In [ ]:
import os
os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")
NEEDED = ["API_KEY", "BASE_URL", "EMBEDDING_BASE_URL"]
missing = [name for name in NEEDED if globals()[name].startswith("여기에") or not globals()[name].strip()]
if missing:   # 임베딩 주소가 없을 때 채팅 주소로 대신하지 않는다
    raise ValueError(", ".join(missing) + "에 안내받은 값을 설정 셀의 따옴표 안에 입력한 뒤 다시 실행하세요. (모델을 호출하기 전에 멈췄습니다)")

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

llm = ChatOpenAI(model=MODEL_NAME, api_key=API_KEY, base_url=BASE_URL, timeout=60, max_retries=1)            # 채팅 주소
json_llm = ChatOpenAI(model=MODEL_NAME, api_key=API_KEY, base_url=BASE_URL, timeout=60, max_retries=1,
                      model_kwargs={"response_format": {"type": "json_object"}})   # JSON 객체 하나만 답하는 모델
embeddings = OpenAIEmbeddings(model=EMBEDDING_MODEL_NAME, api_key=API_KEY, base_url=EMBEDDING_BASE_URL,   # 임베딩 주소
                              check_embedding_ctx_length=False)
print("채팅 모델:", MODEL_NAME, "· 임베딩 모델:", EMBEDDING_MODEL_NAME)

## 1. 문서 준비(01과 같은 방법을 한 셀로)

PDF 8번째 페이지 본문 → 200자·겹침 60자 조각 → Chroma 색인. 이미 색인이 있으면 다시 저장하지 않습니다.

In [ ]:
import os, urllib.request
from pathlib import Path

PDF_URL = "https://www.pwc.com/kr/ko/insights/industry-focus/samilpwc_semicon-guidebook.pdf"
PDF_FILE = Path("data/samilpwc_semicon-guidebook.pdf")       # 원본은 함께 배포하지 않음 → 없으면 내려받음
SOURCE = "삼일PwC 『반도체 산업 Business Guidebook』(2025.12)"
PAGE_INDEX = 7                       # 파이썬은 0부터 셈 → PDF 뷰어의 8번째 페이지
PDF_PAGE, BOOK_PAGE = PAGE_INDEX + 1, 6   # 책자에 인쇄된 쪽 번호는 6
VIEW_URL = f"{PDF_URL}#page={PDF_PAGE}"   # #page=8은 뷰어 이동용. 코드에서는 PAGE_INDEX로 페이지를 고른다

def ensure_pdf():
    if not PDF_FILE.is_file():
        PDF_FILE.parent.mkdir(parents=True, exist_ok=True)
        try:
            with urllib.request.urlopen(PDF_URL, timeout=60) as r:     # 기본 요청으로 받는다(브라우저 흉내 헤더는 거절됨)
                PDF_FILE.write_bytes(r.read())
            print("발행처에서 내려받음:", PDF_URL)
        except Exception as e:
            raise RuntimeError(f"PDF를 받지 못했습니다({type(e).__name__}). 브라우저로 {PDF_URL} 을 내려받아 "
                               f"이 노트북 옆 {PDF_FILE} 에 두고 다시 실행하세요.")
    return PDF_FILE

from pypdf import PdfReader

def page_runs(page):
    """페이지의 글자 묶음마다 (글자 크기, 세로 위치, 글자)를 모은다."""
    runs = []
    page.extract_text(visitor_text=lambda t, cm, tm, fd, fs: runs.append((round(fs * (tm[0] or 1), 1), tm[5], t)))
    return runs

def read_body(page):
    """제목(20pt)과 본문(10pt, 공정 그림 아래쪽)만 남기고 그림 속 짧은 이름표는 뺀다."""
    runs = page_runs(page)
    title = "".join(t for size, y, t in runs if size >= 19).replace("\n", "").strip()
    body = "".join(t for size, y, t in runs if 9.5 <= size <= 10.5 and y < 360).replace("\n", "").strip()
    return title, body

from langchain_core.documents import Document

def split_text(text, size, overlap):
    """size 글자마다 자르고, 다음 조각은 앞 조각의 끝 overlap 글자부터 시작한다 → (시작, 끝) 목록"""
    assert 0 <= overlap < size, "겹침은 0 이상, 조각 크기보다 작아야 합니다"
    spans, start = [], 0
    while True:
        end = min(start + size, len(text))
        spans.append((start, end))
        if end >= len(text):
            return spans
        start += size - overlap

def make_chunks(title, body, size, overlap):
    return [Document(page_content=body[s:e], metadata={
                "chunk_id": f"pwc-p{PDF_PAGE}-c{i:02d}", "source": SOURCE, "title": title, "file": PDF_FILE.name,
                "book_page": BOOK_PAGE, "pdf_page": PDF_PAGE, "url": VIEW_URL, "start": s, "end": e,
                "chunk_size": size, "overlap": overlap})
            for i, (s, e) in enumerate(split_text(body, size, overlap), start=1)]

import hashlib
from langchain_chroma import Chroma

def open_index(chunks):
    """설정(조각 크기·겹침)·본문·임베딩 모델이 같으면 같은 컬렉션 이름 → 다시 실행해도 중복 저장하지 않는다."""
    m = chunks[0].metadata
    key = EMBEDDING_MODEL_NAME + "".join(c.page_content for c in chunks)
    name = f"pwc-p{PDF_PAGE}-{m['chunk_size']}-{m['overlap']}-{hashlib.sha256(key.encode('utf-8')).hexdigest()[:8]}"
    store = Chroma(collection_name=name, embedding_function=embeddings, persist_directory="chroma_db",
                   collection_metadata={"hnsw:space": "cosine"})            # 거리 = 코사인 거리(작을수록 가까움)
    ids = [c.metadata["chunk_id"] for c in chunks]
    have = set(store.get(ids=ids)["ids"])
    new = [c for c in chunks if c.metadata["chunk_id"] not in have]
    if new:
        store.add_documents(new, ids=[c.metadata["chunk_id"] for c in new])
    print(f"컬렉션 {name} · 새로 추가 {len(new)}개 · 저장된 조각 {len(store.get()['ids'])}개")
    return store

def search(store, question, k=3, **kw):
    found = store.similarity_search_with_score(question, k=k, **kw)
    return [{"rank": i, "distance": round(float(d), 4), "text": doc.page_content, **doc.metadata}
            for i, (doc, d) in enumerate(found, start=1)]

def show_hits(hits):
    for h in hits:
        print(f"{h['rank']}위 [{h['chunk_id']}] 거리 {h['distance']} · {h['source']} 책자 {h['book_page']}쪽 = PDF {h['pdf_page']}번째 페이지"
              f" · 본문 {h['start']}~{h['end']}자")
        print("    " + h["text"])

reader = PdfReader(str(ensure_pdf()))
TITLE, BODY = read_body(reader.pages[PAGE_INDEX])
CHUNKS = make_chunks(TITLE, BODY, 200, 60)
store = open_index(CHUNKS)
print(f"문서: {SOURCE} · 책자 {BOOK_PAGE}쪽(PDF {PDF_PAGE}번째 페이지) · ‘{TITLE}’ · 본문 {len(BODY)}자 · 조각 {len(CHUNKS)}개")

## 2. 답변 모델에 줄 지시문과 메시지

검색된 원문만 근거로 답하고, 문장마다 **조각 ID와 원문 그대로의 인용구**를 붙이게 합니다. 모델이 붙인 인용구는 코드가 원문과 다시 대조합니다(모델의 말을 그대로 믿지 않음).

In [ ]:
RAG_SYSTEM = """당신은 반도체 공정 자료를 안내하는 도우미입니다. 한국어로 3~4문장 이내로 쉽게 답합니다.
[검색된 원문]에 있는 내용만 근거로 답하고, 원문에 없는 사실은 쓰지 않습니다.
JSON 객체 하나만 출력합니다.
{"answerable": true 또는 false,
 "sentences": [{"text": "답변 문장", "chunk_id": "근거 조각 ID", "quote": "그 조각 원문에서 글자 그대로 옮긴 짧은 구절"}],
 "missing": ["원문으로 확인할 수 없는 질문의 부분"]}
- 답변 문장마다 그 문장을 뒷받침하는 조각 ID 하나와 인용구를 붙입니다. 인용구는 원문을 바꾸지 말고 그대로 옮깁니다.
- 질문 일부만 원문으로 답할 수 있으면 답할 수 있는 문장만 쓰고, 나머지는 missing에 적습니다.
- 원문으로 답할 수 없으면 answerable을 false, sentences를 []로 둡니다."""

def format_context(hits):
    return "\n\n".join(f"[조각ID: {h['chunk_id']}] {h['source']} 책자 {h['book_page']}쪽(PDF {h['pdf_page']}쪽)\n{h['text']}" for h in hits)

def make_message(question, hits):
    return f"[질문]\n{question}\n\n[검색된 원문]\n{format_context(hits)}"

import json

def read_json(text):
    text = str(text).strip()
    a, b = text.find("{"), text.rfind("}")
    try:
        return json.loads(text[a:b + 1]) if a >= 0 and b > a else {}
    except json.JSONDecodeError:
        return {}

def squash(text):
    return "".join(str(text).split())

def check_citations(data, hits):
    """모델이 댄 조각 ID가 검색 결과에 있고, 인용구가 그 조각 원문에 글자 그대로 있는지 코드로 확인한다."""
    by_id = {h["chunk_id"]: h for h in hits}
    rows = []
    for s in data.get("sentences", []):
        h = by_id.get(s.get("chunk_id"))
        ok = bool(h) and bool(s.get("quote")) and squash(s["quote"]) in squash(h["text"])
        rows.append({**s, "verified": ok, "source": (f"{h['source']} 책자 {h['book_page']}쪽 = PDF {h['pdf_page']}번째 페이지"
                                                       if h else "검색 결과에 없는 조각"), "url": h["url"] if h else ""})
    return rows

from IPython.display import HTML, display

def ask(question, k=3):
    hits = search(store, question, k=k)
    print("① 검색된 원문 ──────────────"); show_hits(hits)
    message = make_message(question, hits)
    print("\n② LLM에 전달되는 질문과 근거(사람 메시지 그대로) ──────────────"); print(message)
    reply = llm.invoke([("system", RAG_SYSTEM), ("human", message)])
    data = read_json(reply.content)
    print("\n③ 답변(모델 응답 JSON) ──────────────"); print(json.dumps(data, ensure_ascii=False, indent=1))
    rows = check_citations(data, hits)
    print("\n④ 인용문과 실제 출처(코드 확인) ──────────────")
    if not rows:
        print("근거 문장 없음 →", "제공된 문서에서 확인할 수 없습니다." if not data.get("answerable") else "(모델이 근거를 대지 않음)")
    for i, r in enumerate(rows, start=1):
        mark = "확인됨" if r["verified"] else "확인 실패 — 출처를 붙이지 않음"
        print(f"[{i}] {r.get('text')}\n    인용 “{r.get('quote')}” · {r.get('chunk_id')} · {r['source']} · {mark}")
    for m in data.get("missing", []):
        print("문서로 확인할 수 없는 부분:", m)
    if any(r["verified"] for r in rows):
        display(HTML(f'<a href="{hits[0]["url"]}" target="_blank">원문 PDF 해당 페이지 열기(책자 {BOOK_PAGE}쪽 = PDF {PDF_PAGE}번째 페이지)</a>'))
    return data, rows

## 3. 08에서 찾은 원문으로 답하기 — 문서에 근거가 있는 질문

질문: **감광액이 덮이지 않은 부분을 없애는 단계는?**

08의 마지막 검색과 같은 질문입니다. 출력은 ① 08에서 찾은 원문(검색 결과) → ② LLM에 보낸 메시지 → ③ 답변 → ④ 직접 인용과 쪽수 순서로 읽습니다.

In [ ]:
data_ok, rows_ok = ask("감광액이 덮이지 않은 부분을 없애는 단계는?")

## 4. 관련 문단은 검색되지만 답할 수 없는 질문

질문: **우리 회사 A라인의 어제 식각 불량 원인은 뭐야?**

식각을 설명한 조각은 검색됩니다. 하지만 특정 회사·라인·날짜의 불량 원인은 이 쪽에 없습니다. 검색 결과가 있다는 것과 그 결과로 답할 수 있다는 것은 다릅니다.

In [ ]:
data_out, rows_out = ask("우리 회사 A라인의 어제 식각 불량 원인은 뭐야?")

## 5. 영상·그림 사례 — 질문 → 자막·캡션 검색 → 재생 링크 또는 그림

같은 순서를 영상 자막과 기사 그림에도 적용합니다. 검색 결과의 **metadata에서** 재생 시각 링크와 그림 URL을 꺼내 표시합니다.

영상: [MIT OCW 6.774 Lecture 20 Etching – Introduction](https://www.youtube.com/watch?v=4q5k1hLfyrM) (자막: 같은 영상을 임베드한 [MIT 공식 WebVTT](https://ocw.mit.edu/courses/6-774-physics-of-microfabrication-front-end-processing-fall-2004/1qv-RkuGGz7mFmOaR3Kq1JxAKb6ThWhhd_transcript.webvtt), 영상 설명의 이용 조건 CC BY-NC-SA — 기업 교육에서 자막을 재사용할 때는 허용 범위를 확인) · 그림: [삼성반도체 「반도체 8대 공정 5탄 ‘식각 공정’」](https://semiconductor.samsung.com/kr/support/tools-resources/fabrication-process/eight-essential-semiconductor-fabrication-processes-part-5-etching-a-circuit-pattern/) 본문 그림의 캡션과 대체 텍스트. 임베딩에는 **글자(자막·캡션·대체 텍스트)만** 넣습니다. 이미지 자체를 분석하거나 검색하지 않습니다.

In [ ]:
import html, re, urllib.request
from langchain_core.documents import Document

VIDEO_ID = "4q5k1hLfyrM"
VIDEO_TITLE = "MIT OCW 6.774 Lecture 20: Etching – Introduction (Judy Hoyt)"
VIDEO_PAGE = f"https://www.youtube.com/watch?v={VIDEO_ID}"
MIT_VTT = ("https://ocw.mit.edu/courses/6-774-physics-of-microfabrication-front-end-processing-fall-2004/"
           "1qv-RkuGGz7mFmOaR3Kq1JxAKb6ThWhhd_transcript.webvtt")          # 같은 영상을 임베드한 MIT 공식 페이지의 영어 자막
ARTICLE_URL = ("https://semiconductor.samsung.com/kr/support/tools-resources/fabrication-process/"
               "eight-essential-semiconductor-fabrication-processes-part-5-etching-a-circuit-pattern/")
ARTICLE_TITLE = "삼성반도체 「반도체 8대 공정 5탄, 반도체 회로패턴의 완성 ‘식각 공정’」"

def fetch(url):
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req, timeout=30) as r:
        return r.read().decode("utf-8", errors="replace")

def seconds(t):                       # "00:02:00.370" → 120.37
    h, m, s = t.split(":")
    return int(h) * 3600 + int(m) * 60 + float(s)

def caption_text():
    """유튜브 자막을 먼저 요청하고, 비어 있거나 실패하면 MIT 공식 WebVTT를 쓴다. (자막 원문, 자막 출처)"""
    try:
        body = fetch(f"https://www.youtube.com/api/timedtext?v={VIDEO_ID}&lang=en&fmt=vtt")
        if "-->" in body:
            return body, "YouTube 자막"
        print("유튜브 직접 자막 조회: 빈 응답 → MIT 공식 WebVTT 사용")
    except Exception as e:
        print(f"유튜브 직접 자막 조회 실패({type(e).__name__}) → MIT 공식 WebVTT 사용")
    return fetch(MIT_VTT), MIT_VTT

def load_video_docs(minutes=15, window=30):
    """앞 minutes분 자막을 window초 묶음으로 → Document(본문=자막, metadata=시작 시각·재생 링크·자막 출처)"""
    try:
        vtt, caption_source = caption_text()
    except Exception as e:
        print(f"자막을 가져오지 못했습니다({type(e).__name__}). 영상 원문: {VIDEO_PAGE}")
        return []
    groups = {}
    for start, text in re.findall(r"(\d\d:\d\d:\d\d\.\d{3}) --> [^\n]*\n(.*?)(?:\n\n|\Z)", vtt, flags=re.S):
        sec = seconds(start)
        text = " ".join(text.split())
        if text and sec < minutes * 60:
            groups.setdefault(int(sec // window) * window, []).append(text)
    return [Document(page_content=" ".join(lines), metadata={
                "media": "video", "doc_id": f"video-{t:04d}s", "start_sec": t, "time": f"{t // 60:02d}:{t % 60:02d}",
                "url": f"{VIDEO_PAGE}&t={t}s", "thumbnail": f"https://i.ytimg.com/vi/{VIDEO_ID}/hqdefault.jpg",
                "title": VIDEO_TITLE, "caption_source": caption_source})
            for t, lines in sorted(groups.items())]

def load_figure_docs():
    """기사 본문 그림의 실제 캡션(▲ …)과 대체 텍스트 → Document(본문=캡션·대체 텍스트, metadata=이미지 URL·기사 출처)"""
    try:
        page = fetch(ARTICLE_URL)
    except Exception as e:
        print(f"기사를 가져오지 못했습니다({type(e).__name__}). 기사 원문: {ARTICLE_URL}")
        return []
    docs, seen = [], set()
    for m in re.finditer(r'<img[^>]+src="([^"]+etching-a-circuit-pattern_\d+_pc[^"]*)"[^>]*alt="([^"]*)"', page):
        src, alt = html.unescape(m.group(1)), html.unescape(m.group(2)).strip()
        after = re.sub(r"<[^>]+>", " ", page[m.end():m.end() + 1500])
        cap = re.search(r"▲\s*([^▲<]{2,60}?)\s{2,}", html.unescape(after))
        if src in seen or not cap:            # 같은 그림의 반응형 중복과 캡션 없는 머리 그림은 건너뜀
            continue
        seen.add(src)
        caption = "▲ " + cap.group(1).strip()
        docs.append(Document(page_content=f"그림 캡션: {caption} / 대체 텍스트: {alt}", metadata={
            "media": "figure", "doc_id": f"figure-{len(docs) + 1}", "caption": caption, "alt": alt,
            "image_url": src, "url": ARTICLE_URL, "title": ARTICLE_TITLE}))
    return docs

def open_media_index(docs):
    key = EMBEDDING_MODEL_NAME + "".join(d.page_content for d in docs)
    name = "media-" + hashlib.sha256(key.encode("utf-8")).hexdigest()[:8]
    media_store = Chroma(collection_name=name, embedding_function=embeddings, persist_directory="chroma_db",
                         collection_metadata={"hnsw:space": "cosine"})
    ids = [d.metadata["doc_id"] for d in docs]
    have = set(media_store.get(ids=ids)["ids"]) if ids else set()
    new = [d for d in docs if d.metadata["doc_id"] not in have]
    if new:
        media_store.add_documents(new, ids=[d.metadata["doc_id"] for d in new])
    print(f"영상·그림 컬렉션 {name} · 새로 추가 {len(new)}개 · 저장된 문서 {len(media_store.get()['ids'])}개")
    return media_store

def media_search(question, media, k=2):
    """media="video" 또는 "figure"로 메타데이터 필터를 걸고 검색한다."""
    if not MEDIA_DOCS:
        print("영상·그림 자료를 가져오지 못해 이 셀은 건너뜁니다. 위 셀의 원문 링크를 확인하세요.")
        return []
    found = media_store.similarity_search_with_score(question, k=k, filter={"media": media})
    return [{"rank": i, "distance": round(float(d), 4), "text": doc.page_content, **doc.metadata}
            for i, (doc, d) in enumerate(found, start=1)]

def show_media(hits):
    """검색 결과의 metadata에서 재생 링크·그림 URL을 꺼내 화면에 표시한다."""
    from IPython.display import HTML, display
    for h in hits:
        if h["media"] == "video":
            display(HTML(f'<div style="display:flex;gap:10px;align-items:flex-start;margin:6px 0">'
                         f'<img src="{h["thumbnail"]}" width="160" alt="영상 썸네일(식별용)">'
                         f'<div><b>{h["rank"]}위 · {h["time"]}부터</b> (거리 {h["distance"]})<br>{html.escape(h["text"])}<br>'
                         f'<a href="{h["url"]}" target="_blank">▶ {h["time"]}부터 재생</a> · 자막 출처: {html.escape(h["caption_source"])}</div></div>'))
        else:
            display(HTML(f'<div style="margin:6px 0"><b>{h["rank"]}위 · {html.escape(h["caption"])}</b> (거리 {h["distance"]})<br>'
                         f'<img src="{h["image_url"]}" width="420" alt="{html.escape(h["alt"])}"><br>'
                         f'대체 텍스트: {html.escape(h["alt"])} · <a href="{h["url"]}" target="_blank">기사 원문</a></div>'))

MEDIA_DOCS = load_video_docs() + load_figure_docs()
media_store = open_media_index(MEDIA_DOCS) if MEDIA_DOCS else None

def ask_media(question, media):
    hits = media_search(question, media)
    if not hits:
        return
    context = "\n".join(f"[{h['doc_id']}] {h['text']}" for h in hits)
    reply = llm.invoke([("system", "아래 [검색된 자료]만 근거로 한국어 한두 문장으로 답합니다. 자료에 없으면 없다고 답합니다."),
                        ("human", f"[질문]\n{question}\n\n[검색된 자료]\n{context}")])
    print("답:", reply.content)
    show_media(hits)

자막은 영어입니다. 질문에 영어 용어(wet etching)를 함께 적어 검색합니다. 한국어로만 물었을 때 결과가 어떻게 달라지는지는 03에서 비교합니다.

In [ ]:
ask_media("습식 식각(wet etching)을 언급하는 영상 구간은?", media="video")

In [ ]:
ask_media("플라즈마 생성 과정을 보여주는 그림은?", media="figure")

## 정리

| 확인한 것 | 근거가 있는 질문 | 답할 수 없는 질문 |
|---|---|---|
| 검색 | 식각 문장을 담은 조각 반환 | 식각 관련 조각은 반환됨 |
| 답변 | 원문 문장에 기대어 답함 | 문서에서 확인할 수 없다고 답함 |
| 출처 | 인용구가 원문에 있는지 코드로 확인 → 책자 6쪽·PDF 8번째 페이지 | 확인된 인용 없음 → 출처를 붙이지 않음 |

영상·그림도 검색 대상은 글자(자막·캡션)이고, 화면에 보여 주는 링크·그림은 검색 결과의 metadata에서 나옵니다. 03에서는 질문 표현과 검색 조건을 바꿔 결과가 어떻게 달라지는지 비교합니다.